# 卖家表第一步：粒度、主键、地区字段和关联覆盖
商品阶段的三个理解题已经核对。本课认识卖家主数据，继续区分“属性表”和“销售明细表”。

| 表 | 一行代表什么 | 用哪个字段识别记录 |
| --- | --- | --- |
| olist_sellers_dataset.csv | 一个卖家的属性记录 | seller_id |
| olist_order_items_dataset.csv | 一笔订单中的一条商品明细 | order_id + order_item_id |

seller_id 在卖家表里应该唯一，在商品明细里可以多次出现。
卖家表没有订单金额、销量、评分或发货耗时，这些要从相应业务表计算。

本课先检查原始值、在副本里规范地区格式、检查明细覆盖。下一步再做关联与金额核对。


## 1. 独立读取卖家表
seller_zip_code_prefix 是卖家邮编的前五位，不是完整邮编。它按字符串保存，保留前导零。

地区字段表示来源记录中的卖家地区；不能只凭这个字段确认每条商品实际从哪里发货。


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# 从项目根目录或 notebooks 目录打开都可以。
project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从 olist-operations-analytics 项目目录打开本课')

data_dir = project_dir.parent / 'archive'

# 原表保留字符串。把空字段识别为缺失，邮编前导零不会丢失。
sellers_raw = pd.read_csv(
    data_dir / 'olist_sellers_dataset.csv',
    dtype=str, encoding='utf-8-sig',
    keep_default_na=False, na_values=['']
)

print('卖家表行数、列数：', sellers_raw.shape)
display(sellers_raw.head())


卖家表行数、列数： (3095, 4)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


## 2. 检查主键与缺失
nunique 统计不同非空值数；duplicated 统计首条之后重复出现的行数。
主键既要唯一，也要非空。城市、州、邮编相同并不表示是同一个卖家，不能按地址直接去重。


In [2]:
seller_key_check = pd.Series({
    '总行数': len(sellers_raw),
    'seller_id不同值数': sellers_raw['seller_id'].nunique(),
    'seller_id缺失数': sellers_raw['seller_id'].isna().sum(),
    'seller_id非空但只有空白数': (
        sellers_raw['seller_id'].notna()
        & sellers_raw['seller_id'].str.strip().eq('')
    ).sum(),
    'seller_id重复行数_首条之后': sellers_raw['seller_id'].duplicated().sum(),
    '整行重复数_首条之后': sellers_raw.duplicated().sum()
})
display(seller_key_check)

# 分开看真正的缺失、空白文本、不同值数量。
# 例如州不同值只有 23，并不代表卖家表只有 23 个卖家。
field_check = pd.DataFrame({
    '原始缺失数': sellers_raw.isna().sum(),
    '非空但只有空白数': sellers_raw.apply(
        lambda col: (col.notna() & col.str.strip().eq('')).sum()
    ),
    '不同值数': sellers_raw.nunique()
})
display(field_check)

# 主键异常时停止，先检查原因；不擅自删除或补造 ID。
if seller_key_check[
    ['seller_id缺失数', 'seller_id非空但只有空白数',
     'seller_id重复行数_首条之后']
].ne(0).any():
    raise ValueError('卖家主键需要核查，先不要关联')


总行数                   3095
seller_id不同值数         3095
seller_id缺失数             0
seller_id非空但只有空白数        0
seller_id重复行数_首条之后       0
整行重复数_首条之后               0
dtype: int64

,原始缺失数,非空但只有空白数,不同值数
seller_id,0,0,3095
seller_zip_code_prefix,0,0,2246
seller_city,0,0,611
seller_state,0,0,23


## 3. 在副本中规范地区格式
与客户表类似：去首尾空白，城市小写、州缩写大写。当前数据可能本来就符合这些格式，
变化数为 0 也是有效检查结果。不会因为“做了清洗”就必须改动数据。

不补零、不截断邮编，不去掉城市中的标点，也不模糊合并城市。
相同城市标签出现在多个州，可能存在同名或记录问题，后续结合地理表核查。


In [3]:
# copy() 使下面的规范操作作用在工作副本上，原始表保持原值。
sellers_clean = sellers_raw.copy()

location_cols = ['seller_zip_code_prefix', 'seller_city', 'seller_state']
for col in location_cols:
    sellers_clean[col] = sellers_clean[col].str.strip()

sellers_clean['seller_city'] = sellers_clean['seller_city'].str.lower()
sellers_clean['seller_state'] = sellers_clean['seller_state'].str.upper()

# 检查哪些字段发生变化。当前文件预期都是 0。
changed_counts = pd.Series({
    col: (
        sellers_raw[col].notna()
        & sellers_raw[col].ne(sellers_clean[col])
    ).sum()
    for col in location_cols
}, name='规范后变化记录数')
display(changed_counts)


seller_zip_code_prefix    0
seller_city               0
seller_state              0
Name: 规范后变化记录数, dtype: int64

## 4. 邮编、州和城市检查
五位数字只说明邮编前缀格式通过，不代表邮编与州、城市的对应关系已核实。
州缩写除格式检查外，复用客户阶段保存的 IBGE 代码参考；本格不需要联网。

异常标记用于保留核查线索。本课不会把带标记的卖家或明细全部删除。


In [4]:
reference_path = project_dir / 'docs' / 'references' / 'ibge_state_codes_v1.json'
state_codes = set(json.loads(reference_path.read_text(encoding='utf-8'))['state_codes'])

# fullmatch 要求整个字符串符合规则。na=False 使缺失不算通过格式检查。
sellers_clean['flag_zip_format_invalid'] = (
    ~sellers_clean['seller_zip_code_prefix'].str.fullmatch('[0-9]{5}', na=False)
)
sellers_clean['flag_state_format_invalid'] = (
    ~sellers_clean['seller_state'].str.fullmatch('[A-Z]{2}', na=False)
)
# AA 可能符合两位大写字母格式，但不在参考代码中，所以另查取值。
sellers_clean['flag_state_unknown'] = ~sellers_clean['seller_state'].isin(state_codes)
sellers_clean['flag_city_missing'] = (
    sellers_clean['seller_city'].isna() | sellers_clean['seller_city'].eq('')
)

seller_location_flags = [
    'flag_zip_format_invalid', 'flag_state_format_invalid',
    'flag_state_unknown', 'flag_city_missing'
]
display(sellers_clean[seller_location_flags].sum())

print('以0开头的邮编前缀记录数：',
      sellers_clean['seller_zip_code_prefix'].str.startswith('0', na=False).sum())

# .loc 只挑选样本显示，不会改写邮编。
display(sellers_clean.loc[
    sellers_clean['seller_zip_code_prefix'].str.startswith('0', na=False),
    ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
].head())


flag_zip_format_invalid      0
flag_state_format_invalid    0
flag_state_unknown           0
flag_city_missing            0
dtype: int64

以0开头的邮编前缀记录数： 1027


,seller_id,seller_zip_code_prefix,seller_city,seller_state
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
8,768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP
12,8bd0f31cf0a614c658f6763bd02dea69,01222,sao paulo,SP
13,05a48cc8859962767935ab9087417fbb,05372,sao paulo,SP
19,f9ec7093df3a7b346b7bcf7864069ca3,05138,sao paulo,SP


## 5. 预检查销售明细中的卖家覆盖
本格先读需要的三个字段，检查能否按 seller_id 找到卖家表。
实际关联和商品金额、运费核对留到下一步，不用在此处重新清洗金额。

注意方向：明细中的卖家找不到属性表，和属性表中某个卖家没有销售明细，是两个不同问题。
前者需要先核查；后者不一定是数据错误，也不应直接删除卖家属性。


In [5]:
items_keys = pd.read_csv(
    data_dir / 'olist_order_items_dataset.csv',
    usecols=['order_id', 'order_item_id', 'seller_id'],
    dtype=str, encoding='utf-8-sig',
    keep_default_na=False, na_values=['']
)

seller_coverage = pd.Series({
    '商品明细行数': len(items_keys),
    '明细涉及订单数': items_keys['order_id'].nunique(),
    '明细涉及不同卖家数': items_keys['seller_id'].nunique(),
    '明细seller_id缺失数': items_keys['seller_id'].isna().sum(),
    '明细非空seller_id找不到卖家表的行数': (
        items_keys['seller_id'].notna()
        & ~items_keys['seller_id'].isin(sellers_clean['seller_id'])
    ).sum(),
    '卖家表中没有销售明细的卖家数': (
        ~sellers_clean['seller_id'].isin(items_keys['seller_id'])
    ).sum()
})
display(seller_coverage)

# 这里只演示一个卖家在明细中出现多次。
# 相同 seller_id 不是重复明细；仍要用订单ID+明细序号识别销售记录。
example_seller_id = items_keys['seller_id'].value_counts().index[0]
display(items_keys.loc[
    items_keys['seller_id'].eq(example_seller_id),
    ['order_id', 'order_item_id', 'seller_id']
].head())


商品明细行数                    112650
明细涉及订单数                    98666
明细涉及不同卖家数                   3095
明细seller_id缺失数                 0
明细非空seller_id找不到卖家表的行数         0
卖家表中没有销售明细的卖家数                 0
dtype: int64

,order_id,order_item_id,seller_id
23,001021efaa8636c29475e7734483457d,1,6560211a19b47992c3666cc44a7e94c0
117,00418a49a685c6bb60633291c3fab17a,1,6560211a19b47992c3666cc44a7e94c0
148,005a1dded353107dbabf8ffc83a20365,1,6560211a19b47992c3666cc44a7e94c0
175,006e05e81c068ddc38e48ebe189b922e,1,6560211a19b47992c3666cc44a7e94c0
182,00750f68bb64537034e93554c64b41b5,1,6560211a19b47992c3666cc44a7e94c0


## 我的理解
在下面补充你的回答：

1. 卖家表有多少条记录、多少个不同 seller_id？seller_id 能否作为主键，依据是什么？
卖家表有3095条记录，有3095个不同的sell_id，可以作为主键，非空且唯一。
2. 为什么邮编前缀要按字符串读取？如果多个卖家城市和邮编相同，能否按地址去重？
因为有些城市的邮编是以0开始的，按照数字读取会忽略0，所以必须安扎字符串读取；不能去重，城市和邮编相同不能代表是同一个卖家。
3. 卖家表的一行、商品明细表的一行分别代表什么？下一步按 seller_id 关联时，为什么应该用 many_to_one？
卖家表的一行代表一个卖家的信息、商品明细表的一行代表的是一笔订单中的一个商品明细；因为一个sell_id可能与多个商品关联。
4. 本课的“找不到卖家表的明细行数”为 0，是否就能说明卖家邮编、城市、州的地理对应关系全部正确？为什么？
商品表“找不到卖家表的明细行数”为 0只能说明所有商品都找到了对应的卖家，不能说明卖家邮编、城市、州的地理对应关系全部正确。
你完成并反馈后，再继续商品明细关联与金额控制。本课没有计算卖家绩效排名或卖家差评率。
